# SkyGuard: Weather-Station Sensor Anomaly Detection
## INT396 — Unsupervised Learning Project

**Academic Term:** 2024–2025  
**Domain:** Environmental Telemetry Analytics / Unsupervised Anomaly Detection  
**Primary Algorithm:** Isolation Forest (with LOF and DBSCAN comparative benchmarks)  

---

### Problem Statement
Automated weather stations collect continuous environmental telemetry (temperature, precipitation, wind speed). Sensor degradation, debris blockage, and transmission artifacts can corrupt observations. In the absence of verified sensor failure ground truth, this notebook demonstrates how **unsupervised machine learning** can isolate anomalous observations based purely on structural statistical atypicality.

### 1. Environment Setup & Library Imports

In [1]:
import sys
from pathlib import Path

# Add project root to sys.path
project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_loader import load_raw_dataset, audit_physical_consistency, generate_data_quality_report
from src.feature_engineering import compute_engineered_features, get_feature_list
from src.preprocessing import WeatherPreprocessor
from src.train import train_isolation_forest, load_trained_model
from src.evaluate import evaluate_contamination_sensitivity, compare_unsupervised_algorithms
from src.utils import set_academic_plot_style

set_academic_plot_style()
print("Environment initialized successfully. Python version:", sys.version.split()[0])

### 2. Data Ingestion & Physical Consistency Audit
We ingest `weather_data_world_2025-01-01_2025-02-28.csv` (6,077 observations across 103 cities and 35 countries).
Strings with unit annotations (`°C`, `mm`, `km/h`, `hPa`) are parsed into clean numeric floats, and coordinates are split into `latitude` and `longitude`.

In [2]:
raw_df = load_raw_dataset(parse_measurements=True)
print(f"Dataset Shape: {raw_df.shape}")
print(f"Unique Stations / Cities: {raw_df['City'].nunique()}")
print(f"Date Range: {raw_df['DATE'].min()} to {raw_df['DATE'].max()}")
raw_df[['City', 'Country', 'DATE', 'TMIN', 'TAVG', 'TMAX', 'PRCP', 'WSPD']].head()

#### Meteorological Physical Constraints Audit
We evaluate thermodynamic laws across all observations:  
1. $T_{MIN} \le T_{MAX}$ (0 violations).  
2. $T_{MIN} \le T_{AVG} \le T_{MAX}$ (Identified **7 physical anomalies**!).

In [3]:
audited_df, physical_summary = audit_physical_consistency(raw_df)
print("Physical Consistency Audit Summary:")
for k, v in physical_summary.items():
    print(f"  - {k}: {v}")

inconsistent = audited_df[audited_df['is_physically_inconsistent']]
inconsistent[['City', 'Country', 'DATE', 'TMIN', 'TAVG', 'TMAX', 'inconsistency_reason']]

### 3. Exploratory Data Analysis & Statistical Profiling
Examine descriptive statistics, skewness, and extreme ranges across all baseline features.

In [4]:
features = ['TMIN', 'TAVG', 'TMAX', 'PRCP', 'WSPD']
stats = raw_df[features].describe().T
stats['skewness'] = raw_df[features].skew()
stats[['count', 'mean', 'std', 'min', '50%', 'max', 'skewness']].round(2)

#### Correlation Matrix Heatmap
Verify collinearity among temperature features and independence of precipitation and wind.

In [5]:
plt.figure(figsize=(7, 5))
corr = raw_df[features].corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, vmin=-1, vmax=1)
plt.title("Weather Feature Correlation Matrix (Baseline)")
plt.tight_layout()
plt.show()

### 4. Domain Meteorological Feature Engineering
We engineer four domain-justified features:  
1. **Diurnal Temperature Range ($DTR = T_{MAX} - T_{MIN}$)**  
2. **Diurnal Midpoint Departure ($\Delta T = T_{AVG} - \frac{T_{MAX} + T_{MIN}}{2}$)**  
3. **Log-Precipitation ($\log(1 + PRCP)$)**: Dampens skewness from 5.72 to 1.73  
4. **City-Relative Temperature Departure ($T_{AVG} - \overline{T_{\text{city}}}$)**

In [6]:
df_eng = compute_engineered_features(raw_df)
print("Engineered Features Created:")
df_eng[['City', 'DATE', 'TEMP_RANGE', 'TEMP_DEVIATION', 'PRCP_LOG', 'TEMP_DEV_CITY']].head()

### 5. Leakage-Free Preprocessing Pipeline
We use `WeatherPreprocessor`, which fits missing value imputation (medians) strictly on training records and standardizes variables without data leakage.

In [7]:
preprocessor = WeatherPreprocessor(feature_mode='baseline', scaler_type='standard')
X_scaled = preprocessor.fit_transform(raw_df)
print(f"Preprocessed Feature Matrix Shape: {X_scaled.shape}")
print(f"Missing values remaining: {np.isnan(X_scaled).sum()}")
print(f"Mean of scaled features (approx 0): {X_scaled.mean(axis=0).round(3)}")
print(f"Std of scaled features (approx 1): {X_scaled.std(axis=0).round(3)}")

### 6. Model Training: Isolation Forest
We train the primary Isolation Forest model ($n\_estimators=150, contamination=0.03, random\_state=42$) and compute continuous decision function scores.

In [8]:
model, preprocessor, results_df, metadata = train_isolation_forest(
    df=raw_df,
    contamination=0.03,
    n_estimators=150,
    random_state=42,
    save_artifacts=True
)

flagged_count = int(results_df['anomaly'].sum())
print(f"Total records trained: {metadata['total_records_trained']:,}")
print(f"Flagged observations: {flagged_count:,} ({metadata['flagged_percentage']:.2f}%)")
print(f"Mean anomaly score: {metadata['score_statistics']['mean']:.4f}")

#### Empirical Anomaly Score Distribution
Negative scores denote anomalous records; positive scores denote normal observations.

In [9]:
plt.figure(figsize=(9, 4.5))
sns.histplot(results_df[~results_df['anomaly']]['anomaly_score'], color='#2b5c8f', kde=True, label='Normal Observations', bins=40, stat='density')
sns.histplot(results_df[results_df['anomaly']]['anomaly_score'], color='#d9534f', kde=True, label='Flagged Anomalies', bins=20, stat='density')
plt.axvline(0, color='black', linestyle='--', label='Decision Threshold (0.0)')
plt.title('Isolation Forest: Anomaly Score Distribution')
plt.xlabel('Anomaly Decision Function Score')
plt.ylabel('Density')
plt.legend()
plt.tight_layout()
plt.show()

### 7. Contamination Sensitivity Analysis
Evaluate model response across contamination parameters: 1%, 3%, 5%, and 10%.

In [10]:
sensitivity_df = evaluate_contamination_sensitivity(raw_df, contamination_levels=[0.01, 0.03, 0.05, 0.10])
sensitivity_df[['contamination_setting', 'records_analyzed', 'flagged_records', 'flagged_percentage', 'mean_anomaly_score', 'runtime_ms']]

### 8. Multi-Paradigm Comparative Benchmark
Benchmark Isolation Forest against Local Outlier Factor (LOF) and DBSCAN using Silhouette scores and execution latency.

In [11]:
comp_df, overlap_info = compare_unsupervised_algorithms(raw_df, contamination=0.03)
print("Algorithm Comparison Benchmark Table:")
display(comp_df)
print("\nSet Overlap & Jaccard Similarity:")
for k, v in overlap_info.items():
    print(f"  - {k}: {v}")

### 9. Qualitative Anomaly Investigation
Examine the meteorological reality behind the most anomalous observations (Severe polar vortex, heavy rain, gale storms).

In [12]:
top_anomalies = results_df[results_df['anomaly']].sort_values('anomaly_score').head(10)
display_cols = ['City', 'Country', 'DATE', 'TMIN', 'TAVG', 'TMAX', 'PRCP', 'WSPD', 'anomaly_score', 'severity']
top_anomalies[display_cols]

### 10. Conclusions & Academic Limitations

1. **Unsupervised Rigor:** Without verified ground-truth sensor-failure annotations, we deliberately avoid reporting supervised metrics (accuracy, precision, recall, F1-score) to prevent scientific fabrication.
2. **Algorithm Superiority:** Isolation Forest achieves a superior silhouette separation score (**0.5518**) compared to Local Outlier Factor (**0.2246**), with fast sub-150ms execution latency.
3. **Physical Anomaly Discovery:** The pipeline successfully identified 7 physical inconsistencies in real telemetry where daily average temperatures violated thermodynamic boundaries.
4. **Production Readiness:** The serialized model and preprocessor enable instant, zero-retraining inference in the accompanying Streamlit dashboard.